# Gharas Saudi Dataset Collection

## Import Libraries

In [ ]:
import os
import json
from dotenv import load_dotenv
import pandas as pd
import json_repair
import base64
from google import genai
from google.genai import types
import pathlib
import httpx
from pydantic import BaseModel
import enum


load_dotenv(os.path.join('..','.env'))

## Helper Functions

In [2]:
def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

## Configuration

In [ ]:
class Config:
    # Define project directories
    base_dir = '/content/drive/MyDrive/Gharas-Saudi-Project'
    docs_dir = os.path.join(base_dir, 'docs')
    output_dir = os.path.join(base_dir, 'output')
    os.makedirs(output_dir, exist_ok=True)

    # Define file paths
    # 'Riyadh-Plants-Manual-Ar.pdf','Riyadh-Plants-Manual-English.pdf'
    file_path = os.path.join(docs_dir, 'Riyadh-Plants-Manual-English.pdf')
    output_file = os.path.join(output_dir, 'plants_dataset.json')


    # Define model configuration
    gemini_api_key = os.getenv('GEMINI_API_KEY')
    model_name = "gemini-1.5-pro-latest"
    temperature = 0.0
    seed = 42



# ............. Create Config .............
cfg = Config()

## Initi Client Gemini API

In [5]:
client = genai.Client(api_key=cfg.gemini_api_key)

## Load Documents

In [6]:
files = [
    client.files.upload(file=cfg.file_path)
]

## Plant Pydantic

In [7]:
# =======================
# ## Plant Pydantic
# =======================
from typing import List
from pydantic import BaseModel, Field
import enum

class ClimateCategory(str, enum.Enum):
    VERY_LOW  = "Very Low"
    LOW       = "Low"
    MODERATE  = "Moderate"
    HIGH      = "High"
    EXTREME   = "Extreme"

class PlantType(str, enum.Enum):
    TREE                 = "Tree"
    SHRUB                = "Shrub"
    SUCCULENT            = "Succulent Plant"
    SUCCULENT_SHRUB      = "Succulent Shrub"
    HERBACEOUS_PERENNIAL = "Herbaceous perennial"
    ELEGANT_PLANT        = "Elegant Plant"


class SoilType(str, enum.Enum):
    SANDY        = "Sandy"
    SANDY_LOAM   = "Sandy-Loam"
    CALCAREOUS   = "Calcareous"
    GYPSIFEROUS  = "Gypsiferous"
    SALINE       = "Saline"
    CLAY_LOAM    = "Clay-Loam"
    SILT_LOAM    = "Silt-Loam"

class Plant(BaseModel):
    name:        str                      = Field(..., description="Common English name")
    type:        PlantType                = Field(..., description="Growth form of the plant")
    temperature: List[ClimateCategory]    = Field(..., description="Temperature tolerance categories")
    humidity:    List[ClimateCategory]    = Field(..., description="Humidity tolerance categories")
    precip:      List[ClimateCategory]    = Field(..., description="Precipitation tolerance categories")
    soil:        List[SoilType]           = Field(..., description="Preferred soil types")


## Prompt

In [8]:
# ================
# ## Prompts
# ================
system_instruction = """
You are an expert in agronomy, botany, and plant data mining,
gathering a master dataset to power a plant recommendation engine in Saudi Arabia.
The engine will match the user's monthly climate (temperature, humidity, rainfall)
and prevailing soil type with plants that can grow in those conditions.

Your task:
• Work from the attached “Riyadh Plants Manual” PDF.
• Identify **every** plant in the manual (≥ 300).
• Populate every field of the Plant schema—name, type, temperature,
  humidity, precip, soil—using the allowed enum values; leave nothing
  blank or null.
• Map all free-text climate and soil descriptors to the five
  ClimateCategory levels (Very Low → Extreme) and the SoilType enum.
• Output the data only; no comments, formatting, or explanatory text.
""".strip()


prompt = """
Extract the complete plant dataset as instructed and provide the data only.
""".strip()


## Generate Content Config

In [9]:
generate_content_config = types.GenerateContentConfig(
        temperature=cfg.temperature,
        seed=cfg.seed,
        response_mime_type="application/json",
        response_schema =list[Plant]
    )

## Contents

In [10]:
contents = [
      types.Part.from_uri(file_uri=files[0].uri,mime_type=files[0].mime_type),
      system_instruction,
    #   prompt
      ]


## Generate Content

In [11]:
response = client.models.generate_content(
    model=cfg.model_name,
    contents=contents,
    config=generate_content_config,
)

## Extract Response

In [12]:
response.text

'[{"name": "Salt Wattle", "type": "Tree", "temperature": ["Moderate"], "humidity": ["Very Low", "Low"], "precip": ["Low"], "soil": ["Saline", "Clay-Loam"]}, {"name": "Cuthbertson\'s Wattle", "type": "Shrub", "temperature": ["Low"], "humidity": ["Very Low", "Low"], "precip": ["Low"], "soil": ["Sandy", "Calcareous"]}, {"name": "Salam Acacia", "type": "Tree", "temperature": ["Low"], "humidity": ["Very Low"], "precip": ["Low"], "soil": ["Sandy", "Calcareous"]}, {"name": "Arad Acacia", "type": "Tree", "temperature": ["Moderate"], "humidity": ["Very Low", "Low"], "precip": ["Low"], "soil": ["Calcareous", "Sandy"]}, {"name": "Sweet Acacia", "type": "Tree", "temperature": ["Low"], "humidity": ["Very Low", "Low"], "precip": ["Low"], "soil": ["Sandy"]}, {"name": "Grey-haired Acacia", "type": "Tree", "temperature": ["Very Low"], "humidity": ["Very Low"], "precip": ["Low"], "soil": ["Sandy", "Calcareous"]}, {"name": "Sweet Thorn", "type": "Tree", "temperature": ["Moderate"], "humidity": ["Very Low

In [13]:
output_json = parse_json(response.text)
len(output_json),output_json[0]

(161,
 {'name': 'Salt Wattle',
  'type': 'Tree',
  'temperature': ['Moderate'],
  'humidity': ['Very Low', 'Low'],
  'precip': ['Low'],
  'soil': ['Saline', 'Clay-Loam']})

In [15]:
output_plants: list[Plant] = parse_json(response.text)
len(output_plants),output_plants[0]

(161,
 {'name': 'Salt Wattle',
  'type': 'Tree',
  'temperature': ['Moderate'],
  'humidity': ['Very Low', 'Low'],
  'precip': ['Low'],
  'soil': ['Saline', 'Clay-Loam']})

In [22]:
df = pd.DataFrame(output_json).dropna()
df

,name,type,temperature,humidity,precip,soil
0,Salt Wattle,Tree,[Moderate],"[Very Low, Low]",[Low],"[Saline, Clay-Loam]"
1,Cuthbertson's Wattle,Shrub,[Low],"[Very Low, Low]",[Low],"[Sandy, Calcareous]"
2,Salam Acacia,Tree,[Low],[Very Low],[Low],"[Sandy, Calcareous]"
3,Arad Acacia,Tree,[Moderate],"[Very Low, Low]",[Low],"[Calcareous, Sandy]"
4,Sweet Acacia,Tree,[Low],"[Very Low, Low]",[Low],[Sandy]
...,...,...,...,...,...,...
155,Jerusalem Thorn,Tree,[Low],"[Low, Moderate, High]",[Low],"[Sandy, Calcareous]"
156,Bahia Grass,Herbaceous perennial,[Low],"[Low, Moderate, High]",[High],"[Sandy, Clay-Loam]"
157,Umbrella Plant,Herbaceous perennial,[Low],"[Moderate, High]",[High],"[Sandy, Clay-Loam]"
158,Reed,Herbaceous perennial,[Very Low],"[Low, Moderate]",[High],"[Sandy, Clay-Loam, Saline]"


In [23]:
df.shape

(160, 6)

## Save

In [29]:
df.to_json(cfg.output_file,orient='records',indent=2)

In [31]:
df.to_csv(f"{cfg.output_file.split('.')[0]}.csv",index=False)